In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import os
import re
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
PROJECT_ROOT = Path.cwd().parent
RAG_OUTPUT = PROJECT_ROOT / "data" / "processed" / "career_mentor"
MODEL_PATH = PROJECT_ROOT / "models"
RAG_OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_PATH.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("RAG Output:", RAG_OUTPUT)
print("Notebook working directory:", Path.cwd())

Project: /Users/rizwansalim/Desktop/SkillBridgeByZequa
RAG Output: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor
Notebook working directory: /Users/rizwansalim/Desktop/SkillBridgeByZequa/notebooks


In [3]:
# Set a saved candidate_id here to choose a particular resume.
# Leave as None to automatically use the newest profile in the Resume Library.
SELECTED_RESUME_ID = None
RESUME_LIBRARY = PROJECT_ROOT / "data" / "processed" / "cv_builder" / "resume_library"
LEGACY_CANDIDATE_FILE = PROJECT_ROOT / "data" / "processed" / "cv_builder" / "updated_candidate.json"

resume_files = sorted(RESUME_LIBRARY.glob("*.json"), key=lambda path: path.stat().st_mtime, reverse=True) if RESUME_LIBRARY.exists() else []
candidate_file = None
if SELECTED_RESUME_ID:
    selected_file = RESUME_LIBRARY / f"{SELECTED_RESUME_ID}.json"
    if selected_file.is_file():
        candidate_file = selected_file
    else:
        raise FileNotFoundError(f"Resume ID not found: {SELECTED_RESUME_ID}")
elif resume_files:
    candidate_file = resume_files[0]
elif LEGACY_CANDIDATE_FILE.is_file():
    candidate_file = LEGACY_CANDIDATE_FILE
else:
    raise FileNotFoundError(f"No candidate profile found. Run Notebook 09 first. Expected folder: {RESUME_LIBRARY}")

with candidate_file.open("r", encoding="utf-8") as file:
    candidate_data = json.load(file)
print("Candidate profile loaded:", candidate_file.name)
print("Available fields:", list(candidate_data.keys()))

Candidate profile loaded: candidate-sample-resume-2.json
Available fields: ['personal_details', 'education', 'experience', 'projects', 'skills', 'certifications', 'languages']


In [4]:
# Notebook 09 stores fields at the top level. This also supports the older nested format.
personal = candidate_data.get("personal_details", {})
if not isinstance(personal, dict):
    personal = {}

mentor_profile = {
    "name": candidate_data.get("name") or personal.get("full_name") or "Candidate",
    "email": candidate_data.get("email") or personal.get("email", ""),
    "phone": candidate_data.get("phone") or personal.get("phone", ""),
    "skills": candidate_data.get("skills", []),
    "education": candidate_data.get("education", []),
    "experience": candidate_data.get("experience", []),
    "projects": candidate_data.get("projects", []),
    "summary": candidate_data.get("summary", "")
}

# Convert text sections into lists for consistent prompt formatting.
for section_name in ["skills", "education", "experience", "projects"]:
    section_value = mentor_profile[section_name]
    if isinstance(section_value, str):
        mentor_profile[section_name] = [section_value] if section_value.strip() else []
    elif section_value is None:
        mentor_profile[section_name] = []
    elif not isinstance(section_value, list):
        mentor_profile[section_name] = [str(section_value)]

print("Candidate:", mentor_profile["name"])
print("Skills:", len(mentor_profile["skills"]))
print("Education entries:", len(mentor_profile["education"]))
print("Experience entries:", len(mentor_profile["experience"]))
print("Project entries:", len(mentor_profile["projects"]))

Candidate: Candidate
Skills: 5
Education entries: 0
Experience entries: 0
Project entries: 0


In [5]:
print("Candidate profile summary prepared.")
print("Candidate:", mentor_profile["name"])
print("Skill count:", len(mentor_profile["skills"]))
print("Education entries:", len(mentor_profile["education"]))
print("Experience entries:", len(mentor_profile["experience"]))
print("Project entries:", len(mentor_profile["projects"]))

Candidate profile summary prepared.
Candidate: Candidate
Skill count: 5
Education entries: 0
Experience entries: 0
Project entries: 0


In [6]:
profile_path = RAG_OUTPUT / "mentor_candidate_profile.json"

with open(profile_path, "w", encoding="utf-8") as file:
    json.dump(mentor_profile, file, indent=4, ensure_ascii=False)

print("Saved:", profile_path)
print("File exists:", profile_path.exists())

Saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor/mentor_candidate_profile.json
File exists: True


In [7]:
occupations_path = PROJECT_ROOT / "data" / "processed" / "unified_occupations.csv"
skills_path = PROJECT_ROOT / "data" / "processed" / "unified_skills.csv"

if not occupations_path.is_file():
    raise FileNotFoundError(f"Occupation dataset not found: {occupations_path}")
if not skills_path.is_file():
    raise FileNotFoundError(f"Skill dataset not found: {skills_path}")

occupations_df = pd.read_csv(occupations_path)
skills_df = pd.read_csv(skills_path)
if len(occupations_df.columns) < 2 or len(skills_df.columns) < 2:
    raise ValueError("Occupation and skill CSV files must each contain at least two columns.")
print("Occupations:", occupations_df.shape)
print("Skills:", skills_df.shape)
print("Occupation columns:", occupations_df.columns.tolist())
print("Skill columns:", skills_df.columns.tolist())

Occupations: (4055, 3)
Skills: (13949, 3)
Occupation columns: ['Occupation ID', 'Occupation Name', 'Source']
Skill columns: ['Skill ID', 'Skill Name', 'Source']


In [8]:
display(occupations_df.head())
display(skills_df.head())

,Occupation ID,Occupation Name,Source
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,ESCO
1,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,ESCO
2,http://data.europa.eu/esco/occupation/0019b951...,precision device inspector,ESCO
3,http://data.europa.eu/esco/occupation/0022f466...,air traffic safety technician,ESCO
4,http://data.europa.eu/esco/occupation/002da35b...,hospitality revenue manager,ESCO


,Skill ID,Skill Name,Source
0,http://data.europa.eu/esco/skill/0005c151-5b5a...,manage musical staff,ESCO
1,http://data.europa.eu/esco/skill/00064735-8fad...,supervise correctional procedures,ESCO
2,http://data.europa.eu/esco/skill/000709ed-2be5...,apply anti-oppressive practices,ESCO
3,http://data.europa.eu/esco/skill/0007bdc2-dd15...,control compliance of railway vehicles regulat...,ESCO
4,http://data.europa.eu/esco/skill/00090cc1-1f27...,identify available services,ESCO


In [9]:
# Select the ID and display-name columns. This supports common ESCO and project column names.
occupation_id_column = "conceptUri" if "conceptUri" in occupations_df.columns else occupations_df.columns[0]
occupation_name_column = "preferredLabel" if "preferredLabel" in occupations_df.columns else ("occupation_name" if "occupation_name" in occupations_df.columns else occupations_df.columns[1])
skill_id_column = "conceptUri" if "conceptUri" in skills_df.columns else skills_df.columns[0]
if "preferredLabel" in skills_df.columns:
    skill_name_column = "preferredLabel"
elif "Skill Name" in skills_df.columns:
    skill_name_column = "Skill Name"
elif "skill_name" in skills_df.columns:
    skill_name_column = "skill_name"
else:
    skill_name_column = skills_df.columns[1]

career_documents = []
for _, row in occupations_df.iterrows():
    occupation_id = str(row[occupation_id_column])
    occupation_name = str(row[occupation_name_column])
    career_documents.append({"document_id": occupation_id, "category": "occupation", "title": occupation_name,
                             "content": f"Career: {occupation_name}. Occupation identifier: {occupation_id}."})
for _, row in skills_df.iterrows():
    skill_id = str(row[skill_id_column])
    skill_name = str(row[skill_name_column])
    career_documents.append({"document_id": skill_id, "category": "skill", "title": skill_name,
                             "content": f"Professional skill: {skill_name}. This skill may be relevant to different occupations."})
print("Career documents created:", len(career_documents))
if career_documents:
    print("Sample document:", career_documents[0])

Career documents created: 18004
Sample document: {'document_id': 'http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34', 'category': 'occupation', 'title': 'technical director', 'content': 'Career: technical director. Occupation identifier: http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34.'}


In [10]:
knowledge_base_path = RAG_OUTPUT / "career_knowledge_base.json"

with open(knowledge_base_path, "w", encoding="utf-8") as file:
    json.dump(career_documents, file, indent=4, ensure_ascii=False)

print("Knowledge base saved:", knowledge_base_path)
print("File exists:", knowledge_base_path.exists())

Knowledge base saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor/career_knowledge_base.json
File exists: True


In [11]:
relations_path = PROJECT_ROOT / "data" / "processed" / "esco_occupation_skill_relationships.csv"
if relations_path.is_file():
    relations_df = pd.read_csv(relations_path)
    print("Relationship records:", len(relations_df))
    print("Columns:", relations_df.columns.tolist())
    display(relations_df.head())
else:
    warnings.warn("ESCO relationship CSV not found. Occupation documents will have no linked skill list.")
    relations_df = pd.DataFrame(columns=["occupationLabel", "skillLabel"])

Relationship records: 126051
Columns: ['occupationUri', 'occupationLabel', 'relationType', 'skillType', 'skillUri', 'skillLabel']


,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,knowledge,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques
1,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals
2,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/271a36a0-bc7a...,write risk assessment on performing arts produ...
3,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/47ed1d37-971b...,coordinate with creative departments
4,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/591dd514-735b...,adapt to artists' creative demands


In [12]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+#.\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print(clean_text("Data Scientist — Python, Machine Learning & SQL!"))

data scientist python machine learning sql


In [13]:
# Normalize relationship column names used by the source CSV.
if "occupationLabel" not in relations_df.columns:
    relations_df = relations_df.rename(columns={"occupation_name": "occupationLabel"}) if "occupation_name" in relations_df.columns else relations_df.assign(occupationLabel="")
if "skillLabel" not in relations_df.columns:
    relations_df = relations_df.rename(columns={"skill_name": "skillLabel"}) if "skill_name" in relations_df.columns else relations_df.assign(skillLabel="")

occupation_skill_groups = relations_df.groupby("occupationLabel")["skillLabel"].apply(
    lambda values: values.dropna().astype(str).tolist()
).to_dict()
enriched_documents = []
for _, row in occupations_df.iterrows():
    occupation_name = str(row[occupation_name_column])
    related_skills = occupation_skill_groups.get(occupation_name, [])
    skill_text = ", ".join(sorted(set(related_skills)))
    content = clean_text(f"Career: {occupation_name}. Related professional skills: {skill_text}.")
    enriched_documents.append({"document_id": str(row[occupation_id_column]), "category": "occupation",
                               "title": occupation_name, "content": content})
print("Enriched occupation documents:", len(enriched_documents))
if enriched_documents:
    print("Sample:", enriched_documents[0])

Enriched occupation documents: 4055
Sample: {'document_id': 'http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34', 'category': 'occupation', 'title': 'technical director', 'content': 'career technical director. related professional skills adapt designers work to the performance venue adapt to artists creative demands coordinate technical teams in artistic productions coordinate with creative departments negotiate health and safety issues with third parties organise rehearsals promote health and safety theatre techniques write risk assessment on performing arts production write technical riders.'}


In [14]:
enriched_path = RAG_OUTPUT / "cleaned_career_documents.json"

with open(enriched_path, "w", encoding="utf-8") as file:
    json.dump(enriched_documents, file, indent=4, ensure_ascii=False)

print("Saved:", enriched_path)
print("File exists:", enriched_path.exists())

Saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor/cleaned_career_documents.json
File exists: True


In [15]:
def create_chunks(text, chunk_size=80, overlap=15):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

sample_text = enriched_documents[0]["content"]

print(create_chunks(sample_text))

['career technical director. related professional skills adapt designers work to the performance venue adapt to artists creative demands coordinate technical teams in artistic productions coordinate with creative departments negotiate health and safety issues with third parties organise rehearsals promote health and safety theatre techniques write risk assessment on performing arts production write technical riders.']


In [16]:
career_chunks = []
for document in enriched_documents:
    document_parts = create_chunks(document["content"])
    for chunk_number, chunk_text in enumerate(document_parts):
        career_chunks.append({"chunk_id": f'{document["document_id"]}_{chunk_number}',
                              "document_id": document["document_id"], "category": document["category"],
                              "title": document["title"], "text": chunk_text})
if not career_chunks:
    raise ValueError("No text chunks were created. Check the occupation dataset.")
print("Total documents:", len(enriched_documents))
print("Total chunks:", len(career_chunks))

Total documents: 4055
Total chunks: 9534


In [17]:
for chunk in career_chunks[:5]:
    print("\nTitle:", chunk["title"])
    print("Chunk:", chunk["text"])


Title: technical director
Chunk: career technical director. related professional skills adapt designers work to the performance venue adapt to artists creative demands coordinate technical teams in artistic productions coordinate with creative departments negotiate health and safety issues with third parties organise rehearsals promote health and safety theatre techniques write risk assessment on performing arts production write technical riders.

Title: metal drawing machine operator
Chunk: career metal drawing machine operator. related professional skills advise on machinery malfunctions cold drawing processes consult technical resources dies ensure equipment availability ferrous metal processing keep records of work progress manufacture of small metal parts manufacturing of door furniture from metal manufacturing of metal assembly products manufacturing of weapons and ammunition mark processed workpiece metal drawing processes metal forming technologies monitor automated machines m

In [18]:
chunks_path = RAG_OUTPUT / "career_text_chunks.json"

with open(chunks_path, "w", encoding="utf-8") as file:
    json.dump(career_chunks, file, indent=4, ensure_ascii=False)

print("\nChunks saved:", chunks_path)
print("File exists:", chunks_path.exists())


Chunks saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor/career_text_chunks.json
File exists: True


In [19]:
# Set these before importing Transformers so optional TensorFlow support is not loaded.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
from sentence_transformers import SentenceTransformer
print("Sentence Transformers imported successfully!")

Sentence Transformers imported successfully!


In [20]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

Embedding model loaded: all-MiniLM-L6-v2


In [21]:
chunk_texts = [chunk["text"] for chunk in career_chunks]

career_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", career_embeddings.shape)
print("Number of chunks:", len(chunk_texts))

Batches: 100%|██████████| 298/298 [12:40<00:00,  2.55s/it]

Embedding shape: (9534, 384)
Number of chunks: 9534


In [22]:
embedding_path = RAG_OUTPUT / "career_embeddings.npy"

np.save(embedding_path, career_embeddings)

print("Embeddings saved:", embedding_path)
print("File exists:", embedding_path.exists())
print("Embedding dimensions:", career_embeddings.shape[1])

Embeddings saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor/career_embeddings.npy
File exists: True
Embedding dimensions: 384


In [23]:
%pip install faiss-cpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 5.1 MB/s  0:00:01m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [24]:
import faiss

print("FAISS version:", faiss.__version__)

FAISS version: 1.16.0


In [25]:
embedding_dimension = career_embeddings.shape[1]

career_index = faiss.IndexFlatIP(embedding_dimension)

career_index.add(
    career_embeddings.astype("float32")
)

print("FAISS index created!")
print("Total indexed vectors:", career_index.ntotal)
print("Embedding dimensions:", career_index.d)

FAISS index created!
Total indexed vectors: 9534
Embedding dimensions: 384


In [26]:
index_path = RAG_OUTPUT / "career_faiss.index"

faiss.write_index(career_index, str(index_path))

print("FAISS index saved:", index_path)
print("File exists:", index_path.exists())

FAISS index saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor/career_faiss.index
File exists: True


In [27]:
loaded_index = faiss.read_index(str(index_path))

print("Loaded index vectors:", loaded_index.ntotal)
print("Index dimensions:", loaded_index.d)

Loaded index vectors: 9534
Index dimensions: 384


In [28]:
def retrieve_career_context(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = career_index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, index in zip(scores[0], indices[0]):

        if index == -1:
            continue

        chunk = career_chunks[index]

        results.append({
            "title": chunk["title"],
            "category": chunk["category"],
            "text": chunk["text"],
            "similarity": round(float(score), 4)
        })

    return results

print("Career retrieval function created!")

Career retrieval function created!


In [29]:
test_query = "What skills are required to become a Data Scientist?"

search_results = retrieve_career_context(test_query, top_k=5)

for number, result in enumerate(search_results, start=1):
    print(f"\nResult {number}")
    print("Career:", result["title"])
    print("Similarity:", result["similarity"])
    print("Information:", result["text"])


Result 1
Career: Data Scientists
Similarity: 0.7416
Information: career data scientists. related professional skills .

Result 2
Career: Computer and Information Research Scientists
Similarity: 0.6634
Information: career computer and information research scientists. related professional skills .

Result 3
Career: Clinical Data Managers
Similarity: 0.6312
Information: career clinical data managers. related professional skills .

Result 4
Career: data scientist
Similarity: 0.6146
Information: career data scientist. related professional skills hadoop ldap linq mdx n1ql sparql xquery apply blended learning apply for research funding apply research ethics and scientific integrity principles in research activities build recommender systems business analytics business intelligence collect ict data communicate with a non scientific audience computational biology computer simulation conduct research across disciplines create data models data engineering data ethics data mining data models data

In [30]:
test_query_2 = "Which careers require Python and machine learning?"

search_results_2 = retrieve_career_context(
    test_query_2,
    top_k=5
)

for result in search_results_2:
    print(
        result["title"],
        "| Similarity:",
        result["similarity"]
    )

Office Machine Operators, Except Computer | Similarity: 0.5717
Computer Occupations, All Other | Similarity: 0.5545
Data Scientists | Similarity: 0.5319
computer vision engineer | Similarity: 0.5309
Logisticians | Similarity: 0.5142


In [31]:
retrieval_path = RAG_OUTPUT / "semantic_retrieval_test.json"

with open(retrieval_path, "w", encoding="utf-8") as file:
    json.dump(search_results, file, indent=4, ensure_ascii=False)

print("Retrieval results saved:", retrieval_path)
print("File exists:", retrieval_path.exists())

Retrieval results saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor/semantic_retrieval_test.json
File exists: True


In [32]:
def build_career_context(query, top_k=5):

    results = retrieve_career_context(query, top_k=top_k)

    context = ""

    for i, result in enumerate(results, start=1):
        context += (
            f"\nCareer {i}: {result['title']}\n"
            f"Information: {result['text']}\n"
            f"Similarity: {result['similarity']}\n"
        )

    return context, results


print("RAG context builder created!")

RAG context builder created!


In [33]:
def create_mentor_prompt(query):

    context, sources = build_career_context(query)

    candidate_context = f"""
Candidate Name: {mentor_profile['name']}
Candidate Skills: {mentor_profile['skills']}
Candidate Education: {mentor_profile['education']}
Candidate Experience: {mentor_profile['experience']}
Candidate Projects: {mentor_profile['projects']}
"""

    prompt = f"""
You are SkillBridge AI Career Mentor.

Use the candidate profile and retrieved career information
to answer the user's question.

Rules:
1. Give clear and practical career guidance.
2. Personalize advice using the candidate profile.
3. Do not invent qualifications or experience.
4. If the retrieved information is insufficient, say so.
5. Distinguish facts from recommendations.

CANDIDATE PROFILE:
{candidate_context}

RETRIEVED CAREER INFORMATION:
{context}

USER QUESTION:
{query}

Provide a helpful and structured answer.
"""

    return prompt, sources


print("Mentor prompt generator created!")

Mentor prompt generator created!


In [34]:
mentor_question = (
    "What skills should I improve to become a Data Scientist?"
)

mentor_prompt, mentor_sources = create_mentor_prompt(
    mentor_question
)

print(mentor_prompt)


You are SkillBridge AI Career Mentor.

Use the candidate profile and retrieved career information
to answer the user's question.

Rules:
1. Give clear and practical career guidance.
2. Personalize advice using the candidate profile.
3. Do not invent qualifications or experience.
4. If the retrieved information is insufficient, say so.
5. Distinguish facts from recommendations.

CANDIDATE PROFILE:

Candidate Name: Candidate
Candidate Skills: ['computer science', 'deep learning', 'machine learning', 'Science', 'SQL']
Candidate Education: []
Candidate Experience: []
Candidate Projects: []


RETRIEVED CAREER INFORMATION:

Career 1: Data Scientists
Information: career data scientists. related professional skills .
Similarity: 0.7399

Career 2: Clinical Data Managers
Information: career clinical data managers. related professional skills .
Similarity: 0.6419

Career 3: Computer and Information Research Scientists
Information: career computer and information research scientists. related prof

In [35]:
rag_test_path = RAG_OUTPUT / "rag_pipeline_test.json"

rag_test_data = {
    "question": mentor_question,
    "prompt": mentor_prompt,
    "sources": mentor_sources
}

with open(rag_test_path, "w", encoding="utf-8") as file:
    json.dump(rag_test_data, file, indent=4, ensure_ascii=False)

print("RAG pipeline test saved:", rag_test_path)
print("File exists:", rag_test_path.exists())

RAG pipeline test saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor/rag_pipeline_test.json
File exists: True


In [36]:
import subprocess
try:
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True, check=True)
    print("Ollama installed:", result.stdout.strip() or result.stderr.strip())
except FileNotFoundError:
    print("Ollama is not installed or not available in PATH. Retrieval can still be tested.")
except subprocess.CalledProcessError as error:
    print("Ollama version check failed:", error.stderr)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Ollama installed: ollama version is 0.40.16


In [37]:
try:
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=True)
    print(result.stdout)
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    print("Ollama model list is unavailable:", error)

NAME           ID              SIZE      MODIFIED    
llama3.2:3b    a80c4f17acd5    2.0 GB    3 weeks ago    



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [38]:
OLLAMA_MODEL = "llama3.2:3b"

print("Selected model:", OLLAMA_MODEL)

Selected model: llama3.2:3b


In [39]:
try:
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=True)
    print(result.stdout)
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    print("Ollama model list is unavailable:", error)

NAME           ID              SIZE      MODIFIED    
llama3.2:3b    a80c4f17acd5    2.0 GB    3 weeks ago    



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [40]:
import requests
try:
    response = requests.get("http://localhost:11434/api/tags", timeout=5)
    response.raise_for_status()
    print("Ollama server is running.")
    print("Installed models:", [item.get("name") for item in response.json().get("models", [])])
except requests.RequestException as error:
    print("Ollama server is not reachable. Start Ollama before AI answer generation.")
    print("Details:", error)

Ollama server is running.
Installed models: ['llama3.2:3b']


In [41]:
# This cell replaces the original shell command so it works consistently in Jupyter on macOS.
try:
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=True)
    print(result.stdout)
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    print("Ollama model list is unavailable:", error)

NAME           ID              SIZE      MODIFIED    
llama3.2:3b    a80c4f17acd5    2.0 GB    3 weeks ago    



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [42]:
import requests
url = "http://localhost:11434/api/generate"
payload = {"model": OLLAMA_MODEL, "prompt": "Say hello and introduce yourself as SkillBridge AI Career Mentor.",
           "stream": False, "options": {"num_predict": 50}}
try:
    response = requests.post(url, json=payload, timeout=180)
    response.raise_for_status()
    print("AI RESPONSE:")
    print(response.json().get("response", "No response text was returned."))
except requests.RequestException as error:
    print("Ollama test request could not be completed:", error)
    print("Continue with retrieval tests; AI answer generation requires Ollama.")

AI RESPONSE:
Hello!

I'm SkillBridge AI Career Mentor, and I'm here to help you navigate your career journey with the power of artificial intelligence and personalized guidance. My mission is to empower you with the skills, knowledge, and confidence to succeed in your chosen


In [43]:
import requests

def ask_career_mentor(question):
    """Retrieve relevant career information and ask the local Ollama model."""
    prompt, sources = create_mentor_prompt(question)
    payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
               "options": {"temperature": 0.3, "num_predict": 500}}
    response = requests.post("http://localhost:11434/api/generate", json=payload, timeout=600)
    response.raise_for_status()
    result = response.json()
    return {"question": question, "answer": result.get("response", ""), "sources": sources}

print("SkillBridge AI Career Mentor function created!")

SkillBridge AI Career Mentor function created!


In [44]:
question = "How can I become a Data Scientist?"
try:
    mentor_response = ask_career_mentor(question)
    print("QUESTION:", mentor_response["question"])
    print("\nAI CAREER MENTOR ANSWER:")
    print(mentor_response["answer"])
except requests.RequestException as error:
    mentor_response = None
    print("AI answer generation skipped because Ollama is unavailable:", error)

QUESTION: How can I become a Data Scientist?

AI CAREER MENTOR ANSWER:
Hello! As your SkillBridge AI Career Mentor, I'd be happy to help you explore a career as a Data Scientist.

To get started, let's break down the steps to become a Data Scientist:

1. **Build a strong foundation in computer science and statistics**: As you have skills in computer science, deep learning, and machine learning, you're off to a great start. Consider expanding your knowledge in statistics, probability, and linear algebra to complement your existing skills.
2. **Gain practical experience with data analysis and visualization**: Data Scientists work with various tools and technologies, such as Python, R, SQL, and data visualization libraries like Matplotlib, Seaborn, or Tableau. Practice working with datasets, exploring different visualization techniques, and creating interactive stories with data.
3. **Develop expertise in a specific domain**: Data Scientists often specialize in a particular industry or do

In [45]:
if mentor_response is not None:
    print("KNOWLEDGE SOURCES:")
    for source in mentor_response["sources"]:
        print(source["title"], "| Similarity:", source["similarity"])
else:
    print("No AI response was generated, so there are no answer-specific sources to display.")

KNOWLEDGE SOURCES:
Data Scientists | Similarity: 0.6808
Computer and Information Research Scientists | Similarity: 0.5936
data scientist | Similarity: 0.5787
Data Warehousing Specialists | Similarity: 0.5603
Clinical Data Managers | Similarity: 0.5513


In [46]:
if mentor_response is not None:
    answer_path = RAG_OUTPUT / "career_mentor_response.json"
    with answer_path.open("w", encoding="utf-8") as file:
        json.dump(mentor_response, file, indent=4, ensure_ascii=False)
    print("Response saved:", answer_path)
else:
    print("Response file not created because Ollama did not return an answer.")

Response saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor/career_mentor_response.json


In [47]:
conversation_history = []

print("Conversation memory initialized!")

Conversation memory initialized!


In [48]:
def chat_with_career_mentor(question):

    mentor_response = ask_career_mentor(question)

    conversation_entry = {
        "message_number": len(conversation_history) + 1,
        "question": question,
        "answer": mentor_response["answer"],
        "sources": mentor_response["sources"]
    }

    conversation_history.append(conversation_entry)

    return mentor_response["answer"]


print("Career mentor chat function created!")

Career mentor chat function created!


In [49]:
if mentor_response is not None:
    answer_1 = chat_with_career_mentor("What skills should I learn for Data Science?")
    print("QUESTION 1:")
    print(answer_1)
    answer_2 = chat_with_career_mentor("Which of those skills should I learn first?")
    print("\nQUESTION 2:")
    print(answer_2)
else:
    print("Conversation examples skipped because Ollama is unavailable.")

QUESTION 1:
Based on the candidate profile and retrieved career information, I'd be happy to provide guidance on what skills to learn for Data Science.

**Data Science Career Overview**

Data Science is a field that combines elements of computer science, statistics, and domain expertise to extract insights from data. As a Data Scientist, you'll work with various types of data, develop predictive models, and communicate insights to stakeholders.

**Key Skills for Data Science**

To excel in Data Science, focus on developing the following skills:

1. **Programming languages**:
	* Python ( NumPy, pandas, scikit-learn, TensorFlow, Keras)
	* R (dplyr, tidyr, caret, ggplot2)
2. **Data manipulation and analysis**:
	* Data cleaning and preprocessing
	* Data visualization (Matplotlib, Seaborn, Plotly)
	* Statistical modeling (regression, hypothesis testing)
3. **Machine learning and deep learning**:
	* Supervised and unsupervised learning
	* Neural networks (TensorFlow, Keras)
	* Model evaluati

In [50]:
for conversation in conversation_history:

    print("\nMessage:", conversation["message_number"])
    print("Question:", conversation["question"])
    print("Answer:", conversation["answer"])


Message: 1
Question: What skills should I learn for Data Science?
Answer: Based on the candidate profile and retrieved career information, I'd be happy to provide guidance on what skills to learn for Data Science.

**Data Science Career Overview**

Data Science is a field that combines elements of computer science, statistics, and domain expertise to extract insights from data. As a Data Scientist, you'll work with various types of data, develop predictive models, and communicate insights to stakeholders.

**Key Skills for Data Science**

To excel in Data Science, focus on developing the following skills:

1. **Programming languages**:
	* Python ( NumPy, pandas, scikit-learn, TensorFlow, Keras)
	* R (dplyr, tidyr, caret, ggplot2)
2. **Data manipulation and analysis**:
	* Data cleaning and preprocessing
	* Data visualization (Matplotlib, Seaborn, Plotly)
	* Statistical modeling (regression, hypothesis testing)
3. **Machine learning and deep learning**:
	* Supervised and unsupervised le

In [51]:
history_path = RAG_OUTPUT / "career_mentor_conversation_history.json"

with open(history_path, "w", encoding="utf-8") as file:
    json.dump(
        conversation_history,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Conversation history saved!")
print("File exists:", history_path.exists())
print("Total conversations:", len(conversation_history))

Conversation history saved!
File exists: True
Total conversations: 2


In [52]:
required_files = {
    "Candidate Profile": RAG_OUTPUT / "mentor_candidate_profile.json",
    "Career Knowledge Base": RAG_OUTPUT / "career_knowledge_base.json",
    "Cleaned Career Documents": RAG_OUTPUT / "cleaned_career_documents.json",
    "Text Chunks": RAG_OUTPUT / "career_text_chunks.json",
    "Embeddings": RAG_OUTPUT / "career_embeddings.npy",
    "FAISS Index": RAG_OUTPUT / "career_faiss.index",
    "Retrieval Results": RAG_OUTPUT / "semantic_retrieval_test.json",
    "RAG Pipeline": RAG_OUTPUT / "rag_pipeline_test.json",
    "Conversation History": RAG_OUTPUT / "career_mentor_conversation_history.json"
}
if mentor_response is not None:
    required_files["AI Mentor Response"] = RAG_OUTPUT / "career_mentor_response.json"

print("SKILLBRIDGE DAY 12 FILE VALIDATION")
print("-" * 50)
all_found = True
for name, path in required_files.items():
    exists = path.exists()
    print(("FOUND " if exists else "MISSING"), "|", name)
    if not exists:
        all_found = False

SKILLBRIDGE DAY 12 FILE VALIDATION
--------------------------------------------------
FOUND  | Candidate Profile
FOUND  | Career Knowledge Base
FOUND  | Cleaned Career Documents
FOUND  | Text Chunks
FOUND  | Embeddings
FOUND  | FAISS Index
FOUND  | Retrieval Results
FOUND  | RAG Pipeline
FOUND  | Conversation History
FOUND  | AI Mentor Response


In [53]:
import numpy as np
import faiss

saved_embeddings = np.load(
    RAG_OUTPUT / "career_embeddings.npy"
)

saved_index = faiss.read_index(
    str(RAG_OUTPUT / "career_faiss.index")
)

print("Embedding shape:", saved_embeddings.shape)
print("FAISS vectors:", saved_index.ntotal)
print("Text chunks:", len(career_chunks))

embedding_valid = (
    saved_embeddings.shape[0] == len(career_chunks)
    and saved_index.ntotal == len(career_chunks)
    and saved_index.d == saved_embeddings.shape[1]
)

print("Embedding and FAISS validation:", embedding_valid)

Embedding shape: (9534, 384)
FAISS vectors: 9534
Text chunks: 9534
Embedding and FAISS validation: True


In [54]:
response_path = RAG_OUTPUT / "career_mentor_response.json"
history_path = RAG_OUTPUT / "career_mentor_conversation_history.json"
if response_path.is_file():
    with response_path.open("r", encoding="utf-8") as file:
        saved_response = json.load(file)
    response_valid = bool(saved_response.get("question")) and bool(saved_response.get("answer"))
else:
    saved_response = None
    response_valid = mentor_response is None  # Retrieval-only mode is an allowed partial run.
with history_path.open("r", encoding="utf-8") as file:
    saved_history = json.load(file)
history_valid = isinstance(saved_history, list)
print("AI response valid or intentionally skipped:", response_valid)
print("Conversation history valid:", history_valid)
print("Conversation count:", len(saved_history))

AI response valid or intentionally skipped: True
Conversation history valid: True
Conversation count: 2


In [55]:
validation_report = {
    "project": "SkillBridgeByZequa",
    "module": "Day 12 - AI Career Mentor + RAG",
    "required_files_found": sum(
        path.exists() for path in required_files.values()
    ),
    "total_required_files": len(required_files),
    "embedding_count": int(saved_embeddings.shape[0]),
    "embedding_dimensions": int(saved_embeddings.shape[1]),
    "faiss_vectors": int(saved_index.ntotal),
    "embedding_validation": bool(embedding_valid),
    "ai_response_validation": bool(response_valid),
    "conversation_history_validation": bool(history_valid),
    "conversation_count": len(saved_history)
}

report_path = RAG_OUTPUT / "day12_validation_report.json"

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(validation_report, file, indent=4)

print("Validation report saved:", report_path)

Validation report saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/career_mentor/day12_validation_report.json


In [56]:
all_checks_passed = (
    all_found
    and embedding_valid
    and response_valid
    and history_valid
)

print("\n" + "=" * 50)
print("SKILLBRIDGE DAY 12 FINAL VALIDATION")
print("=" * 50)

print("Required files:", sum(
    path.exists() for path in required_files.values()
), "/", len(required_files))

print("Embedding validation:", embedding_valid)
print("AI response validation:", response_valid)
print("Conversation history:", history_valid)

if all_checks_passed:
    print("\nVALIDATION PASSED")
else:
    print("\nVALIDATION INCOMPLETE - CHECK MISSING ITEMS")


SKILLBRIDGE DAY 12 FINAL VALIDATION
Required files: 10 / 10
Embedding validation: True
AI response validation: True
Conversation history: True

VALIDATION PASSED
